# Giao diện người dùng tạo sinh kiểm soát

Tính đến thời điểm hiện tại, AI agent của bạn chỉ có thể phản hồi bằng văn bản. Bây giờ, chúng ta sẽ nâng cấp nó để có thể hiển thị các giao diện React như: thẻ thông tin chuyến bay, biểu đồ tròn, hoặc bất kỳ component nào bạn muốn tích hợp. 

Với **GenUI kiểm soát**, bạn sẽ là người xây dựng các component, và AI agent sẽ đóng vai trò quyết định xem nên sử dụng component nào và vào lúc nào.

---

## 📋 Mục tiêu bài học
1. **Hiểu về GenUI** - Khái niệm này là gì và "GenUI kiểm soát" nằm ở đâu trong bức tranh tổng thể.
2. **Đăng ký các frontend component** - Sử dụng hook `useComponent()` để "phơi bày" các React component cho AI agent.
3. **Hiển thị dữ liệu có cấu trúc** - Cho phép agent lựa chọn và điền dữ liệu vào các UI component ngay trong khung chat.

---

## 🚀 Bạn sẽ xây dựng những gì?

Một giao diện chat có khả năng hiển thị các UI component phong phú dựa trên yêu cầu của người dùng:

> **Người dùng:** *Show a flight card for Pacific Air from SFO to JFK departing at 08:30 for $249*

<img src="images/flight-card.png" style="width: 50%; display: block; margin: 0 auto;">

> **Người dùng:** *Please show me the distribution of our revenue by category in a pie chart*

<img src="images/pie-chart.png" style="width: 50%; display: block; margin: 0 auto;">

*(Ghi chú: Biểu đồ được vẽ dựa trên tập dữ liệu CSV thực tế được cung cấp cho backend).*

---

## 🧠 Lý thuyết: GenUI kiểm soát là gì?

**GenUI** là một mô hình nơi các agent phản hồi bằng các giao diện tương tác hoàn chỉnh thay vì chỉ bằng văn bản. **GenUI kiểm soát** là biến thể mang tính kiểm soát cao nhất: Agent **chỉ có thể** render ra những component mà bạn đã đăng ký một cách rõ ràng từ trước.

### Cách thức hoạt động
Mỗi component sau khi được đăng ký sẽ biến thành một "công cụ" cung cấp cho agent với:
- Một cái tên cố định.
- Một schema định kiểu cho đầu vào.
- Một React component được ánh xạ tương ứng.

Agent sẽ không tự sinh ra một giao diện bất kỳ. Thay vào đó, nó truyền dữ liệu có cấu trúc vào các component mà bạn đã xây dựng sẵn trên frontend.

### Ưu và nhược điểm

**✅ Ưu điểm:**
- **Dễ triển khai:** Chỉ cần đăng ký một component là xong.
- **Tính thẩm mỹ cao:** Mọi giao diện hiển thị đều do chính bạn thiết kế và kiểm soát.
- **Độ an toàn cao:** Mô hình AI chỉ có thể gọi các công cụ đã được đăng ký với các tham số đã qua kiểm duyệt.
- **Phù hợp với môi trường thực tế:** Rất tốt cho các ứng dụng có lượng truy cập cao hoặc yêu cầu trải nghiệm người dùng (UX) ổn định, khắt khe.

**❌ Nhược điểm:**
- Khối lượng công việc của frontend sẽ tăng lên theo mỗi tính năng mới (mỗi mẫu giao diện cần một component riêng).
- Thiếu đi sự tự do, linh hoạt so với các dạng GenUI mở.

---

### Hook `useComponent()`
Hook `useComponent` được sử dụng để đăng ký một React component như một công cụ (tool) để Agent có thể gọi bên trong `<CopilotChat />`. Bạn định nghĩa những gì có sẵn, agent sẽ chọn thời điểm sử dụng.

```tsx
useComponent({
  name: "component_name",
  description: "Mô tả để agent biết khi nào nên dùng component này",
  parameters: z.object({ ... }),
  render: MyComponent,
});
```

**Các tham số:**
- `name` *(string, bắt buộc)*: Tên công cụ cung cấp cho mô hình AI.
- `description` *(string, tùy chọn)*: Gợi ý cho mô hình biết khi nào nên gọi công cụ này.
- `parameters` *(Zod schema, tùy chọn)*: Cấu trúc của các props sẽ được truyền vào component.
- `render` *(bắt buộc)*: Một React component (sẽ được render dưới dạng `<Component {...args} />`), hoặc một hàm nhận vào `{ args, status }` nếu bạn muốn tùy chỉnh render (hiển thị trạng thái loading, conditional display,...).

---

## 🛠️ Thực hành từng bước

### Bước 1: Khởi tạo môi trường & Load các API key

Đầu tiên, chúng ta cần nạp các API key để có thể sử dụng mô hình LLM.

In [1]:
from helper import load_api_keys

# Load API key từ file .env
load_api_keys()

✓ OpenAI API key loaded
✓ Google API key loaded


### Bước 2: Khởi động backend agent

Trong bài này, backend được xây dựng bằng **LangGraph** và **FastAPI** (định nghĩa tại file `server.py`). 
Agent này được trang bị một công cụ là `query_data` để đọc dữ liệu doanh thu/chi phí từ file `db.csv`. System prompt của agent được tinh chỉnh để:
1. Luôn gọi `query_data` khi được hỏi về biểu đồ.
2. Gọi công cụ `pieChart` để hiển thị biểu đồ tròn.
3. Gọi công cụ `flightCard` để hiển thị vé máy bay.

In [1]:
# Bắt đầu chạy backend agent ở port 8003
from backend.server import start_backend
start_backend(port=8003)

✓ Server running at http://localhost:8003


### Bước 3: Đăng ký các Frontend component

Đây là bước quan trọng nhất trên frontend (React). Chúng ta sẽ đăng ký 3 component: `showMyName`, `pieChart`, và `flightCard`.

In [2]:
%%writefile frontend/src/App.tsx

import { z } from "zod"
import { CopilotChat } from "@copilotkit/react-core/v2";
import { useComponent } from "@copilotkit/react-core/v2";

import { FlightCard, FlightCardProps } from "@/components/flight-card";
import { PieChart, PieChartProps } from "@/components/pie-chart";

import { useExampleSuggestions } from "@/hooks/use-example-suggestions";

export default function App() {

  // 🪁 Đăng ký component hiển thị tên người dùng
  useComponent({
    name: "showMyName",
    description: "Hiển thị tên của người dùng trong một thẻ.",
    parameters: z.object({ name: z.string() }),
    render: ({ name }) => <div className="bg-blue-500 p-4">Hi, {name}!</div>,
  });

  // 🪁 Đăng ký component pieChart để hiển thị dữ liệu cấu trúc
  useComponent({
    name: "pieChart",
    description: "Hiển thị dữ liệu dưới dạng biểu đồ hình tròn.",
    parameters: PieChartProps,
    render: PieChart,
  });

  // 🪁 Đăng ký component flightCard để hiển thị dữ liệu chuyến bay
  useComponent({
    name: "flightCard",
    description: "Hiển thị thẻ tóm tắt thông tin một chuyến bay.",
    parameters: FlightCardProps,
    render: FlightCard,
  });

  // 🪁 Thêm các gợi ý prompt cho người dùng hiển thị dưới dạng nút bấm
  useExampleSuggestions();

  return <CopilotChat />;

};

Overwriting frontend/src/App.tsx


### Bước 4: Khởi động và hiển thị frontend

Bây giờ chúng ta khởi động frontend dev server và trải nghiệm.

In [ ]:
# Khởi động frontend ở port 3003
from helper import start_frontend, display_app

start_frontend(port=3003)

# Hiển thị UI ngay trên Jupyter Notebook
display_app(port=3003)

Lúc này, agent của bạn đã có 3 giao diện có thể render. Bạn hãy thử nhập vào khung chat các prompt sau:
* *"Show my name as Alex"*
* *"Pie chart"* (Agent sẽ đọc dữ liệu từ CSV và gọi component `pieChart`)
* *"Flight card"* (Agent sẽ gọi component `flightCard`)

---

## 🔍 Khám phá sâu hơn: Tại sao lại dùng Zod?

Bất kỳ React component nào cũng có thể trở thành công cụ GenUI - chỉ cần bạn đăng ký nó với `useComponent()`. Hãy xem qua mã nguồn của `FlightCard` để hiểu rõ cơ chế:

```tsx
import { z } from "zod";

// 1. Định nghĩa Zod schema (vừa là xác thực đầu vào, vừa là hướng dẫn cho LLM)
export const FlightCardProps = z.object({
  title: z.string().describe("Tiêu đề thẻ chuyến bay"),
  airline: z.string().describe("Tên hãng hàng không"),
  origin: z.string().describe("Sân bay hoặc thành phố xuất phát"),
  destination: z.string().describe("Sân bay hoặc thành phố điểm đến"),
  departure_time: z.string().describe("Thời gian khởi hành"),
  price: z.string().describe("Giá vé hiển thị"),
});

// 2. Xuất kiểu dữ liệu TypeScript từ Zod schema
type FlightCardProps = z.infer<typeof FlightCardProps>;

// 3. Xây dựng UI component
export function FlightCard({
  title, airline, origin, destination, departure_time, price,
}: FlightCardProps) {
  return (
    <div className="rounded-lg border bg-white p-3 space-y-2">
      <div className="font-semibold">{title}</div>
      <div className="rounded border p-2 text-sm">
        <div className="font-medium">{airline}</div>
        <div>
          {origin} → {destination}
        </div>
        <div>Departs: {departure_time}</div>
        <div className="font-semibold text-xl mt-2">{price}</div>
      </div>
    </div>
  );
}
```

**Tại sao lại dùng Zod?**
[Zod](https://zod.dev) là thư viện validation và khai báo schema ưu tiên TypeScript (tương tự như [Pydantic](https://docs.pydantic.dev) trong Python). Nó cho phép bạn **chỉ cần định nghĩa props một lần** và dùng chung schema đó cho cả:
1. Định nghĩa type cho React (`type FlightCardProps`).
2. Tham số truyền vào cho AI agent (`parameters` trong `useComponent`).

---

## 🎓 Tổng kết những gì bạn đã học

- **GenUI kiểm soát** giới hạn việc render vào các component đã được đăng ký một cách tường minh, tránh việc AI vẽ ra giao diện lỗi.
- Hook `useComponent()` tạo ra một bản hợp đồng ràng buộc kiểu giữa các đối số của công cụ và props của React component.
- **Tách biệt rạch ròi trách nhiệm:** Backend/agent làm nhiệm vụ chọn công cụ và lấy dữ liệu, trong khi frontend chỉ tập trung vào việc render giao diện.
- Cách tiếp cận này tạo ra trải nghiệm an toàn, dễ dự đoán hơn so với GenUI mở, dù có đánh đổi đôi chút về độ linh hoạt.

> 💡 **Thử thách cho bạn:** 
> Thử tự tạo và đăng ký một Component mới (ví dụ: một Table hoặc một Progress Bar) với `useComponent` và yêu cầu agent sử dụng nó. Bạn sẽ cần định nghĩa một Zod schema cho props và tạo một React component để render chúng.

---

## 📚 Tài liệu tham khảo & Bước tiếp theo

Bạn muốn tìm hiểu sâu hơn? Dưới đây là các tài nguyên hữu ích:
- **[Headless CopilotKit](https://docs.copilotkit.ai/langgraph/custom-look-and-feel/headless-ui)** - Chạy CopilotKit mà không cần dùng UI chat mặc định để làm giao diện tùy biến hoàn toàn.
- **[Frontend Tools Reference](https://docs.copilotkit.ai/langgraph/generative-ui/frontend-tools)** - Toàn bộ tài liệu API của `useComponent()` và các pattern nâng cao.
- **[AG-UI Protocol](https://docs.ag-ui.com)** - Giao thức cốt lõi giúp kết nối CopilotKit với bất kỳ agent backend nào.

👉 **Trong bài tiếp theo:** Chúng ta sẽ chuyển sang **GenUI khai báo** - một phương pháp nằm ở giữa thang đo. Thay vì đăng ký từng component riêng lẻ, bạn sẽ định nghĩa một thư viện các "khối xây dựng" và để AI agent tự do kết hợp chúng thành các layout phức tạp sử dụng tiêu chuẩn A2UI.